# Python + SQL Integration

This notebook demonstrates how to use SQL from Python using the `sqlite3` module. You will learn how to connect to a database, run queries, read results, and write data - all from Python code.

For more details, see the [sqlite3 documentation](https://docs.python.org/3/library/sqlite3.html).

## Setup

In [ ]:
import sqlite3
from pathlib import Path
from urllib.request import urlretrieve

# Download the warehouse database if not present (e.g., on Colab)
db_path = Path("data/warehouse.db")
if not db_path.exists():
    db_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://github.com/olearydj/INSY3010/raw/main/project/warehouse/warehouse.db"
    urlretrieve(url, db_path)
    print("Downloaded warehouse.db")
else:
    print(f"Using {db_path}")

## Connecting to a Database

The `sqlite3` module is included with Python - no installation needed. Working with a database requires two objects: a *connection* and a *cursor*.

In [ ]:
import sqlite3

# create a connection to the database file
con = sqlite3.connect(db_path)

# create a cursor - this is what we use to run queries
cur = con.cursor()

The connection represents the link to the database file. The cursor is the object we use to execute queries and retrieve results.

## Reading Data

### Running a Query

Use `cur.execute()` to run a SQL statement, then retrieve results with `fetchall()`.

In [ ]:
cur.execute("SELECT * FROM Warehouse")
rows = cur.fetchall()

# results are a list of tuples
print(type(rows))
print(type(rows[0]))

In [ ]:
# each tuple is one row from the database
for row in rows:
    print(row)

### Cursor Exhaustion

A cursor is an *iterator* - once you've read through the results, they're gone. This is similar to reading a file: once you reach the end, there's nothing left to read unless you start over.

In [ ]:
cur.execute("SELECT * FROM Carrier")

# loop through results
for row in cur:
    print(row)

In [ ]:
# now try fetchall - the cursor is already exhausted
remaining = cur.fetchall()
print(remaining)  # empty list!

The solution: call `fetchall()` *immediately* after `execute()` and store the results in a variable. Then you can use that variable as many times as you need.

In [ ]:
cur.execute("SELECT * FROM Carrier")
carriers = cur.fetchall()  # store results right away

# now use the list freely
print(f"Found {len(carriers)} carriers")
print(f"First: {carriers[0]}")
print(f"Last: {carriers[-1]}")

### Multi-Line Queries

Use triple-quoted strings for longer queries. This keeps your SQL readable.

In [ ]:
SQL = """
SELECT w.name, COUNT(*) AS shipment_count
FROM Warehouse w
JOIN Shipment s ON w.warehouse_id = s.warehouse_id
GROUP BY w.warehouse_id, w.name
ORDER BY shipment_count DESC
"""

cur.execute(SQL)
results = cur.fetchall()

for row in results:
    print(row)

### Column Names

Query results are tuples - they don't carry column names. Use `cur.description` to get the column names from the most recent query.

In [ ]:
# cur.description holds metadata about the last query
for col_info in cur.description:
    print(col_info[0])  # first element is the column name

### Formatting Output

Combining column names with data makes output more readable.

In [ ]:
# re-run to get fresh results
cur.execute("""
SELECT w.name, w.city, w.state, w.capacity_sqft
FROM Warehouse w
ORDER BY w.capacity_sqft DESC
""")
results = cur.fetchall()

# get column names
columns = []
for col in cur.description:
    columns.append(col[0])
print(columns)

print()
for row in results:
    print(f"  {row[0]} ({row[1]}, {row[2]}) - {row[3]:,} sq ft")

### Parameterized Queries

When a query depends on a variable - like user input or a function argument - use `?` placeholders instead of f-strings. This prevents SQL injection attacks.

In [ ]:
# BAD - never do this (vulnerable to SQL injection)
# category = "Electronics"
# cur.execute(f"SELECT * FROM Product WHERE category = '{category}'")

# GOOD - use ? placeholders
category = "Electronics"
cur.execute("SELECT product_name, unit_weight, unit_cost FROM Product WHERE category = ?",
            (category,))
results = cur.fetchall()

print(f"Products in '{category}':")
for name, weight, cost in results:
    print(f"  {name} - {weight} lbs, ${cost:.2f}")

Note the `(category,)` syntax - the second argument to `execute()` must be a *tuple*, even when there's only one value. The trailing comma makes it a tuple.

In [ ]:
# multiple parameters
min_weight = 5.0
max_cost = 50.0

cur.execute("""
    SELECT product_name, category, unit_weight, unit_cost
    FROM Product
    WHERE unit_weight > ? AND unit_cost < ?
    ORDER BY unit_weight DESC
""", (min_weight, max_cost))

for name, cat, weight, cost in cur.fetchall():
    print(f"  {name} ({cat}) - {weight} lbs, ${cost:.2f}")

### Close the Connection

Always close the connection when you're done.

In [ ]:
con.close()

## Writing Data

So far we've only read from an existing database. You can also create tables and insert data from Python.

### Creating a Table

We'll create a small database from scratch. First, connect to a new file - `sqlite3.connect()` creates the file if it doesn't exist.

In [ ]:
import os

# start fresh each time this section runs
if os.path.exists("data/tutorial.db"):
    os.remove("data/tutorial.db")

con = sqlite3.connect("data/tutorial.db")
cur = con.cursor()

# create a table
cur.execute("CREATE TABLE movie (title TEXT, year INTEGER, score REAL)")

# insert a couple of rows
SQL = """
INSERT INTO movie VALUES
    ('Monty Python and the Holy Grail', 1975, 8.2),
    ('And Now for Something Completely Different', 1972, 7.5)
"""
cur.execute(SQL)

# IMPORTANT: commit saves the changes to the database file
con.commit()

Without `con.commit()`, changes exist only in memory and are lost when the connection closes.

In [ ]:
# verify the data was saved
cur.execute("SELECT * FROM movie")
for row in cur.fetchall():
    print(row)

### Inserting Multiple Rows

Use `executemany()` with `?` placeholders to insert multiple rows from a list. This is both safer (parameterized) and cleaner than building SQL strings by hand.

In [ ]:
more_movies = [
    ("Monty Python Live at the Hollywood Bowl", 1982, 7.9),
    ("Monty Python's The Meaning of Life", 1983, 7.5),
    ("Monty Python's Life of Brian", 1979, 8.0),
]

cur.executemany("INSERT INTO movie VALUES (?, ?, ?)", more_movies)
con.commit()

# verify
cur.execute("SELECT year, title, score FROM movie ORDER BY year")
for year, title, score in cur.fetchall():
    print(f"  {year} - {title} ({score})")

In [ ]:
con.close()

## Context Managers

In the Files & Exceptions lecture, we used `with` statements (context managers) to open files safely - the file is automatically closed even if an error occurs. The same pattern works for database connections.

In [ ]:
# without context manager (what we've been doing)
con = sqlite3.connect(db_path)
cur = con.cursor()
cur.execute("SELECT name FROM Warehouse LIMIT 3")
print(cur.fetchall())
con.close()  # easy to forget!

In [ ]:
# with context manager - connection closes automatically
with sqlite3.connect(db_path) as con:
    cur = con.cursor()
    cur.execute("SELECT name FROM Warehouse LIMIT 3")
    print(cur.fetchall())

# con is already closed here

The `with` pattern is especially useful when writing data. Inside a `with` block, changes are automatically committed if no errors occur, and rolled back if an error is raised.

In [ ]:
with sqlite3.connect("data/tutorial.db") as con:
    cur = con.cursor()
    cur.execute("INSERT INTO movie VALUES ('Spamalot', 2005, 7.0)")
    # no need to call con.commit() - the with block handles it

# verify
with sqlite3.connect("data/tutorial.db") as con:
    cur = con.cursor()
    cur.execute("SELECT title, year FROM movie ORDER BY year DESC LIMIT 3")
    for title, year in cur.fetchall():
        print(f"  {title} ({year})")

## Summary

| Task | Method |
|------|--------|
| Connect to database | `sqlite3.connect(path)` |
| Create a cursor | `con.cursor()` |
| Run a query | `cur.execute(sql)` |
| Get all results | `cur.fetchall()` |
| Get one result | `cur.fetchone()` |
| Column names | `cur.description` |
| Parameterized query | `cur.execute(sql, (val1, val2))` |
| Insert many rows | `cur.executemany(sql, data_list)` |
| Save changes | `con.commit()` |
| Close connection | `con.close()` or use `with` |

---

Auburn University / Industrial and Systems Engineering  
INSY 3010 / Programming and Databases for ISE / Spring 2026  
© Copyright 2026, Danny J. O'Leary.  
For licensing, attribution, and information: [GitHub INSY3010](https://github.com/olearydj/INSY3010)